# Fake Job Posting Detector
NLP: TF-IDF. ML: Naive Bayes vs Logistic Regression. Dataset: Real or Fake Job Posting Prediction (EMSCAD, Kaggle).

## 1. Load data and join text columns

In [1]:
import re, pandas as pd
df = pd.read_csv("fake_job_postings.csv")
print(df.shape); print(df.fraudulent.value_counts())
df["text"] = df.title.fillna("") + " " + df.description.fillna("") + " " + df.requirements.fillna("")

(17880, 18)
fraudulent
0    17014
1      866
Name: count, dtype: int64


## 2. Clean text

In [2]:
def clean(t):
    t = re.sub(r"<[^>]+>", " ", str(t).lower())
    t = re.sub(r"http\S+|#url_\w+#|#email_\w+#|#phone_\w+#", " ", t)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z\s]", " ", t)).strip()
df["text"] = df.text.map(clean)
df = df[df.text.str.len() > 0]
df = df.groupby("fraudulent", group_keys=False).sample(frac=2000/len(df), random_state=42)  # 2,000-row sample
df.text.iloc[0][:300]

'sem coordinator the right candidate will be responsible for coordinating one of the most dynamic ppc accounts in travel industry she he will be responsible for planning developing deploying and reporting of sem campaigns and will be in close contact with the company s management and account manager '

## 3. Split (80:20, stratified) and TF-IDF

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
Xtr, Xte, ytr, yte = train_test_split(df.text, df.fraudulent, test_size=0.2, stratify=df.fraudulent, random_state=42)
tfidf = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2), min_df=3)
Xtr_v = tfidf.fit_transform(Xtr); Xte_v = tfidf.transform(Xte)
Xtr_v.shape

(1600, 20000)

## 4. Train both models and compare

In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
models = {"Naive Bayes": MultinomialNB(alpha=0.1),
          "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", C=10)}
rows = {}
for n, m in models.items():
    m.fit(Xtr_v, ytr); p = m.predict(Xte_v)
    rows[n] = dict(accuracy=accuracy_score(yte, p), precision=precision_score(yte, p), recall=recall_score(yte, p), f1=f1_score(yte, p))
    print(n, "confusion matrix [[TN FP][FN TP]]:", confusion_matrix(yte, p).tolist())
pd.DataFrame(rows).T.round(3)

Naive Bayes confusion matrix [[TN FP][FN TP]]: [[379, 2], [12, 7]]
Logistic Regression confusion matrix [[TN FP][FN TP]]: [[379, 2], [6, 13]]


,accuracy,precision,recall,f1
Naive Bayes,0.965,0.778,0.368,0.500
Logistic Regression,0.980,0.867,0.684,0.765


## 5. Why accuracy alone is not enough

In [5]:
print("Always predict REAL accuracy:", round(1 - yte.mean(), 4))

Always predict REAL accuracy: 0.9525


A model that calls every posting real already scores about 95%. Precision, recall and F1 on the fake class show how many scams are actually caught (recall) and how many real jobs are wrongly flagged (precision).